# Qdrant Product Search Pipeline
**Step by step: CSV → Preprocess → Embed → Store → Retrieve → Evaluate**

## Step 1: Install & Import

In [ ]:
# install if needed
# !pip install pandas sentence-transformers qdrant-client

import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

print('All imports done')

## Step 2: Load CSV
Load product data from Excel/CSV file

In [ ]:
df = pd.read_csv('products.csv')

print(f'Total products: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(5)

## Step 3: Preprocess
- Combine all columns into one text
- Lowercase
- Clean spaces
- Handle missing values

**Note: DO NOT remove stopwords for transformer embeddings**

In [ ]:
def preprocess(row):
    # combine all fields
    fields = [
        str(row.get('Product Name', '')),
        str(row.get('Features', '')),
        str(row.get('Description', '')),
        str(row.get('Tags', ''))
    ]
    # remove NaN values
    fields = [f for f in fields if f != 'nan']
    
    # combine
    text = ' '.join(fields)
    
    # lowercase
    text = text.lower()
    
    # remove extra spaces
    text = ' '.join(text.split())
    
    return text

df['combined_text'] = df.apply(preprocess, axis=1)

print('Sample combined text:')
print(df['combined_text'][0])
print()

print(f'Text length range: {df["combined_text"].str.len().min()} - {df["combined_text"].str.len().max()} chars')

## Step 4: Generate Embeddings
Convert each product text → 384-dimensional vector using all-MiniLM-L6-v2

In [ ]:
# load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded')
print(f'Embedding dimensions: {model.get_sentence_embedding_dimension()}')

# generate embeddings for all products
embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

print(f'\nEmbeddings shape: {embeddings.shape}')
print(f'  {embeddings.shape[0]} products')
print(f'  {embeddings.shape[1]} dimensions per product')
print(f'\nSample vector (first 5 values of product 0):')
print(embeddings[0][:5])

## Step 5: Store in Qdrant
- Create collection with 384 dimensions
- Each product = one Point (vector + payload)
- Payload stores: name, barcode, features, tags

In [ ]:
# connect to Qdrant (in-memory for testing)
client = QdrantClient(':memory:')

# create collection
client.create_collection(
    collection_name='products',
    vectors_config=VectorParams(
        size=384,              # embedding dimensions
        distance=Distance.COSINE  # similarity measure
    )
)
print('Collection created')

# create points
points = []
for idx, row in df.iterrows():
    points.append(PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload={
            'name':        row['Product Name'],
            'barcode':     str(row['Barcode']),
            'features':    str(row['Features']),
            'tags':        str(row['Tags'])
        }
    ))

# store in Qdrant
client.upsert(collection_name='products', points=points)
print(f'Stored {len(points)} products in Qdrant')

# verify
info = client.get_collection('products')
print(f'Collection points count: {info.points_count}')

## Step 6: Retrieve
Search products using natural language queries

In [ ]:
def search(query, top_k=3, threshold=0.3):
    # convert query to vector
    query_vector = model.encode(query).tolist()
    
    # search Qdrant
    results = client.query_points(
        collection_name='products',
        query=query_vector,
        limit=top_k,
        score_threshold=threshold
    ).points
    
    return results

# test queries
queries = [
    'long grain rice',
    'something healthy for breakfast',
    'cleaning product for bathroom',
    'snack for evening',
    'dairy product protein',
    'writing stationery',
]

for query in queries:
    print(f"Query: '{query}'")
    results = search(query)
    for r in results:
        print(f"  [{r.score:.3f}] {r.payload['name']} ({r.payload['barcode']})")
    if not results:
        print('  No results found')
    print()

## Step 7: Evaluate Retrieval Quality
- **Hit Rate**: Did we get at least 1 correct result?
- **Precision@K**: How many returned results are correct?
- **Ground truth**: What SHOULD be returned

In [ ]:
# ground truth
ground_truth = {
    'basmati rice':               ['BR001'],
    'something healthy for breakfast': ['GT001', 'MA001'],
    'cleaning product for bathroom':   ['HC001'],
    'snack for evening':             ['LC001', 'KK001'],
    'dairy product protein':         ['EF001', 'PF001'],
    'writing stationery':            ['NB001', 'PR001'],
}

hit_count      = 0
precision_list = []
total          = len(ground_truth)

print(f'{"─"*60}')
for query, expected in ground_truth.items():
    results   = search(query, top_k=3)
    retrieved = [r.payload['barcode'] for r in results]
    
    # hit rate
    hit = any(b in retrieved for b in expected)
    if hit:
        hit_count += 1
    
    # precision@3 — how many of top 3 are correct
    correct    = sum(1 for b in retrieved if b in expected)
    precision  = correct / len(retrieved) if retrieved else 0
    precision_list.append(precision)
    
    status = '✓ HIT ' if hit else '✗ MISS'
    print(f'{status} | P@3={precision:.2f} | {query}')
    print(f'         Expected : {expected}')
    print(f'         Retrieved: {retrieved}')
    print()

print(f'{"─"*60}')
print(f'Hit Rate    : {hit_count}/{total} = {hit_count/total:.0%}')
print(f'Avg P@3     : {sum(precision_list)/len(precision_list):.2f}')

## Step 8: Improve — Try different thresholds

In [ ]:
print('Threshold experiment:')
print(f'{"─"*40}')

for threshold in [0.7, 0.5, 0.3, 0.2]:
    hits = 0
    for query, expected in ground_truth.items():
        results   = search(query, top_k=3, threshold=threshold)
        retrieved = [r.payload['barcode'] for r in results]
        if any(b in retrieved for b in expected):
            hits += 1
    print(f'Threshold={threshold} → Hit Rate: {hits}/{total} = {hits/total:.0%}')

## Step 9: Visualize Vectors (PCA)
Reduce 384 dimensions → 2D and plot to see how similar products cluster together

## Step 10: Visualize Vectors (t-SNE)
t-SNE gives better clusters than PCA — slower but more accurate visualization

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# category → barcodes mapping
categories = {
    'Grain/Rice':  ['BR001','BR002','BR003','WF001','MF001'],
    'Fruit':       ['AF001','BN001','MN001','OR001','GR001'],
    'Vegetable':   ['TM001','ON001','PT001','CR001','SP001'],
    'Dairy/Meat':  ['MA001','CA001','BA001','PF001','EF001','CB001','ML001'],
    'Snack':       ['LC001','KK001','BG001','BB001','BWB001'],
    'Oil/Cooking': ['SO001','CO001','OO001','TS001','RS001'],
    'Hygiene':     ['DS001','DV001','HS001','CT001','VD001','SE001','HC001'],
    'Beverage':    ['TT001','GT001','NC001'],
    'Other':       ['NB001','PR001','AP001','WB001','UM001','TE001','MG001','PP001'],
}
colors = {
    'Grain/Rice': 'brown', 'Fruit': 'green', 'Vegetable': 'lime',
    'Dairy/Meat': 'blue',  'Snack': 'orange','Oil/Cooking': 'gold',
    'Hygiene': 'purple',   'Beverage': 'cyan','Other': 'gray',
}

def plot_vectors(reduced, title, filename):
    plt.figure(figsize=(14, 10))
    for category, barcodes in categories.items():
        idx = [df[df['Barcode'] == b].index[0] for b in barcodes if b in df['Barcode'].values]
        if not idx: continue
        plt.scatter(reduced[idx, 0], reduced[idx, 1],
                    label=category, color=colors[category], s=120, alpha=0.8)
        for i in idx:
            plt.annotate(df['Product Name'][i], (reduced[i, 0], reduced[i, 1]),
                         fontsize=7, ha='right', va='bottom')
    plt.title(title, fontsize=13)
    plt.legend(loc='upper right', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f'Saved: {filename}')

# ── PCA ───────────────────────────────────────────────────
pca          = PCA(n_components=2)
reduced_pca  = pca.fit_transform(embeddings)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}')
plot_vectors(reduced_pca, 'Product Vectors — PCA\nSimilar products cluster together', 'qdrant_pca.png')

# ── t-SNE ─────────────────────────────────────────────────
print('\nRunning t-SNE (slower but better clusters)...')
tsne         = TSNE(n_components=2, random_state=42, perplexity=10, n_iter=1000)
reduced_tsne = tsne.fit_transform(embeddings)
plot_vectors(reduced_tsne, 'Product Vectors — t-SNE\nBetter clusters than PCA', 'qdrant_tsne.png')